# 🛠️ NL → SolidWorks VBA + Canlı 3D — Tek Tıkla Çalıştır

**Yapacakların (sıra ile her hücrenin sol başındaki ▶ butonuna bas):**
1. **GPU aç:** Üst menü → *Runtime* → *Change runtime type* → **T4 GPU** seç → Save.
2. Aşağıdaki hücreleri yukarıdan aşağıya sırayla çalıştır.
3. 5 numaralı hücrede çıkacak `https://xxxx.gradio.live` linkine tıkla → arayüz açılır.

**Önemli:** 1–5 arası hücreler eğitim YAPMAZ; sistem deterministik fallback model ile zaten çalışır. Eğitim hücresi (6) opsiyoneldir; kaliteyi artırır ama 15–25 dk sürer.

## 1) Dosyaları yükle
Sol paneldeki 📁 ikonuna tıkla, masaüstündeki klasördeki şu 5 dosyayı sürükle-bırak ile yükle:
`data_generator.py`, `ai_model_setup.py`, `visualizer.py`, `export_report.py`, `app.py`


In [ ]:
# 2) Bağımlılıkları kur (~2 dk)
# NOT: numpy<2 ŞART — cadquery 2.4.0 numpy 2.x ile uyumsuz (np.bool8 kaldırıldı)
!pip -q install "numpy<2" cadquery==2.4.0 gradio transformers datasets accelerate sentencepiece
print('OK: kurulum tamam — şimdi Runtime → Restart session yap, sonra hücre 3 ile devam et.')

In [ ]:
# 3) 50.000 örneklik veri setini üret (~30 sn)
!python data_generator.py -n 50000 -o dataset.jsonl

In [ ]:
# 4) HIZLI TEST — fallback model ile bir CAD render'ı dene
from visualizer import show_in_colab
from ai_model_setup import get_model
m = get_model()
pred = m.predict('100x60x10 mm plaka oluştur, dikey kenarlara 5 mm fillet uygula')
print('Op:', pred['op'])
print('--- VBA (ilk 300 char) ---'); print(pred['vba'][:300])
print('--- CadQuery ---'); print(pred['cadquery'])
show_in_colab(pred['cadquery'])

In [ ]:
# 5) GRADIO ARAYÜZÜNÜ BAŞLAT (eğitimsiz, fallback model)
# Çıkan https://xxxx.gradio.live linkine tıkla.
from app import build
build().launch(share=True)

---
## 6) (OPSİYONEL) Modeli fine-tune et — kaliteyi artır

Bunu çalıştırmak için **5 numaralı hücreyi durdur** (Runtime → Interrupt). 15–25 dk sürer.

In [ ]:
# 6a) Smoke test (5 dk) — pipeline çalışıyor mu?
!python ai_model_setup.py --train --subset 500 --epochs 1 --out ./cad_model

In [ ]:
# 6b) Gerçek eğitim (15–25 dk T4) — codet5-small + 5k örnek + 2 epoch
!python ai_model_setup.py --train --subset 5000 --epochs 2 --out ./cad_model

In [ ]:
# 7) AKADEMİK METRİK RAPORU
from ai_model_setup import get_model
from export_report import evaluate
rep = evaluate(get_model('./cad_model'), 'dataset.jsonl',
               sample_size=300, check_validity=True)
print(rep.pretty())

In [ ]:
# 8) Eğitilmiş model ile arayüzü tekrar aç
from app import build
build().launch(share=True)